# IoU 類 loss

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/11-iou-loss/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import math
import torch


def losses(predicted, target):
    assert torch.isfinite(target).all() and (target[2:] > target[:2]).all(), 'GT must have finite coordinates and positive width/height'
    pwh = (predicted[2:]-predicted[:2]).clamp(min=1e-6)
    twh = target[2:]-target[:2]
    inter_wh = (torch.minimum(predicted[2:],target[2:])-torch.maximum(predicted[:2],target[:2])).clamp(min=0)
    intersection = inter_wh.prod()
    union = pwh.prod()+twh.prod()-intersection
    iou = intersection/union.clamp(min=1e-6)
    outer_wh = torch.maximum(predicted[2:],target[2:])-torch.minimum(predicted[:2],target[:2])
    outer_area = outer_wh.prod()
    giou = iou-(outer_area-union)/outer_area.clamp(min=1e-6)
    pc = (predicted[:2]+predicted[2:])/2
    tc = (target[:2]+target[2:])/2
    center_penalty = (pc-tc).square().sum()/outer_wh.square().sum().clamp(min=1e-6)
    v = 4/math.pi**2*(torch.atan(twh[0]/twh[1])-torch.atan(pwh[0]/pwh[1])).square()
    alpha = (v/(1-iou+v).clamp(min=1e-6)).detach()
    return {'iou':1-iou,'giou':1-giou,'diou':1-iou+center_penalty,
            'ciou':1-iou+center_penalty+alpha*v}


def box_from_center(center):
    half_size = torch.tensor([8.,8.],dtype=center.dtype)
    return torch.cat([center-half_size,center+half_size])


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    target = torch.tensor([8.,12.,24.,28.])
    center = torch.nn.Parameter(torch.tensor([40.,20.]))
    initial = losses(box_from_center(center),target)
    assert abs(initial['iou'].item()-1) < 1e-6
    assert abs(initial['giou'].item()-1.2) < 1e-6
    assert abs(initial['diou'].item()-(1+576/1856)) < 1e-6
    assert torch.allclose(initial['diou'],initial['ciou'])
    iou_gradient = torch.autograd.grad(initial['iou'],center,retain_graph=True)[0]
    assert torch.equal(iou_gradient,torch.zeros(2))
    giou_gradient = torch.autograd.grad(initial['giou'],center,retain_graph=True)[0]
    assert torch.allclose(giou_gradient,torch.tensor([.02,0.]),atol=1e-6)
    giou_center = torch.nn.Parameter(torch.tensor([40.,20.]))
    giou_optimizer = torch.optim.SGD([giou_center],lr=50)  # .02 * 50 = 1 pixel left
    giou_initial = losses(box_from_center(giou_center),target)['giou']
    giou_optimizer.zero_grad(set_to_none=True)
    giou_initial.backward()
    assert torch.isfinite(giou_center.grad).all()
    giou_optimizer.step()
    giou_after = losses(box_from_center(giou_center),target)['giou']
    assert torch.allclose(giou_center,torch.tensor([39.,20.]),atol=1e-5)
    assert abs(giou_after.item()-(2-512/624)) < 1e-6 and giou_after < giou_initial
    # float32 cannot store .02 exactly (0.0199999995...); round to print the hand-computed value.
    print('GIoU gradient',[round(value,6) for value in giou_gradient.tolist()],
          'after 1-pixel real SGD move',round(giou_after.item(),6))
    shifted_center = torch.tensor([36.,20.],requires_grad=True)  # 4 pixels left, gap remains 4.
    shifted = losses(box_from_center(shifted_center),target)
    assert shifted['iou'].item() == 1
    assert torch.equal(torch.autograd.grad(shifted['iou'],shifted_center)[0],torch.zeros(2))
    assert abs(shifted['diou'].item()-(1+400/(36**2+16**2))) < 1e-6
    optimizer = torch.optim.SGD([center],lr=100)
    optimizer.zero_grad(set_to_none=True)
    initial['diou'].backward()
    assert torch.isfinite(center.grad).all() and center.grad[0] > 0
    before = center.detach().clone()
    optimizer.step()
    after = losses(box_from_center(center),target)
    assert center[0] < before[0] and after['diou'] < initial['diou']
    print('initial losses',{k:round(v.item(),6) for k,v in initial.items()})
    print('non-overlap plain IoU center gradient',[round(value,6) for value in iou_gradient.tolist()])
    # Pixels get 4 decimals: near 40, adjacent float32 values are ~4e-6 apart, so a 6th decimal is noise.
    print('DIoU center after one real step',[round(value,4) for value in center.detach().tolist()],
          'loss',round(after['diou'].item(),6))
    exact = losses(target,target)
    assert all(abs(value.item()) < 1e-6 for value in exact.values())
    print('exact boxes: all four losses zero')


if __name__ == '__main__':
    main()


GIoU gradient [0.02, 0.0] after 1-pixel real SGD move 1.179487
initial losses {'iou': 1.0, 'giou': 1.2, 'diou': 1.310345, 'ciou': 1.310345}
non-overlap plain IoU center gradient [0.0, 0.0]
DIoU center after one real step [38.7515, 20.0] loss 1.294497
exact boxes: all four losses zero
